# Steam Reviews - PySpark Structured Streaming

Bu notebook, Kafka'dan gelen Steam review verilerini PySpark Structured Streaming ile okuyup
Delta Lake katmanlarına (Bronze → Silver → Gold) yazmaktadır.

**Veri Kaynağı:** Kafka topic `steam-reviews`  
**Hedef:** Delta Lake (Bronze / Silver / Gold)

## BÖLÜM 1 — Spark Session Başlatma

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SteamReviews_StructuredStreaming")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages",
            "io.delta:delta-core_2.12:2.4.0,"
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.streaming.checkpointLocation", "/tmp/checkpoints")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## BÖLÜM 2 — Kafka'dan Stream Okuma

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import from_json, col

# JSON şema tanımı (producer mesaj formatı ile birebir aynı)
review_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("app_id", IntegerType(), True),
    StructField("app_name", StringType(), True),
    StructField("review_text", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_votes", IntegerType(), True),
])

# Kafka'dan stream okuma
raw_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "steam-reviews")
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

# JSON parse
parsed_stream_df = (
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_timestamp")
    .select(
        from_json(col("json_str"), review_schema).alias("data"),
        col("kafka_timestamp")
    )
    .select("data.*", "kafka_timestamp")
)

print("Parsed Stream Schema:")
parsed_stream_df.printSchema()

## BÖLÜM 3 — Veri Temizleme

In [ ]:
from pyspark.sql.functions import length, when, trim

# Temizleme adımları
cleaned_stream_df = (
    parsed_stream_df
    # 1. NULL review_text olanları filtrele
    .filter(col("review_text").isNotNull())
    # 2. review_text uzunluğu < 10 karakter olanları çıkar
    .filter(length(trim(col("review_text"))) >= 10)
    # 3. review_score yalnızca 1 veya -1 olmalı
    .filter(col("review_score").isin(1, -1))
    # 4. Duplicate kayıtları kaldır (app_id + review_text kombinasyonu)
    .dropDuplicates(["app_id", "review_text"])
    # 5. label kolonu ekle: review_score==1 → 1, diğerleri → 0
    .withColumn("label", when(col("review_score") == 1, 1).otherwise(0))
)

print("Cleaned Stream Schema:")
cleaned_stream_df.printSchema()

## BÖLÜM 4 — Delta Lake Katmanlarına Yazma

### 4.1 BRONZE Katmanı — Ham Veri

In [ ]:
# BRONZE: Ham veri, tüm kolonlar
bronze_query = (
    parsed_stream_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/bronze/steam_reviews")
    .option("path", "/delta/bronze/steam_reviews")
    .queryName("bronze_steam_reviews")
    .start()
)

print(f"Bronze query status: {bronze_query.status}")
print(f"Bronze query id: {bronze_query.id}")

### 4.2 SILVER Katmanı — Temizlenmiş Veri

In [ ]:
# SILVER: Temizlenmiş veri, gerekli kolonlar (user_id dahil)
silver_df = cleaned_stream_df.select(
    "timestamp", "user_id", "event_type",
    "app_id", "app_name", "review_text",
    "review_score", "review_votes", "label"
)

silver_query = (
    silver_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/checkpoints/silver/steam_reviews")
    .option("path", "/delta/silver/steam_reviews")
    .queryName("silver_steam_reviews")
    .start()
)

print(f"Silver query status: {silver_query.status}")
print(f"Silver query id: {silver_query.id}")

### 4.3 GOLD Katmanı — Günlük Oyun Başına İstatistikler

In [ ]:
from pyspark.sql.functions import to_date, sum as spark_sum, count, window

# GOLD: Günlük oyun başına pozitif/negatif yorum sayısı
gold_df = (
    cleaned_stream_df
    .withColumn("review_date", to_date(col("timestamp")))
    .groupBy("review_date", "app_id", "app_name")
    .agg(
        count("*").alias("total_reviews"),
        spark_sum(when(col("review_score") == 1, 1).otherwise(0)).alias("positive_count"),
        spark_sum(when(col("review_score") == -1, 1).otherwise(0)).alias("negative_count")
    )
)

gold_query = (
    gold_df
    .writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", "/tmp/checkpoints/gold/daily_stats")
    .option("path", "/delta/gold/daily_stats")
    .queryName("gold_daily_stats")
    .start()
)

print(f"Gold query status: {gold_query.status}")
print(f"Gold query id: {gold_query.id}")

### Streaming Sorgularını İzleme

In [ ]:
import time

# Aktif streaming sorgularını listele
for q in spark.streams.active:
    print(f"Query: {q.name} | Status: {q.status} | ID: {q.id}")

# Bir süre bekle ve sonra stream'leri durdur (test için)
# time.sleep(60)
# bronze_query.stop()
# silver_query.stop()
# gold_query.stop()

---
## BATCH MODE — Streaming Yerine Alternatif

Aşağıdaki hücreler, Kafka'dan batch modda (bir kerelik) veri okuyup Delta Lake'e yazmak için kullanılır.
Streaming yerine test/debug amaçlı çalıştırılabilir.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BATCH MODE - Kafka'dan batch okuma
# ═══════════════════════════════════════════════════════════════════

from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import from_json, col, length, trim, when, to_date, count
from pyspark.sql.functions import sum as spark_sum

# JSON şema tanımı (producer mesaj formatı ile birebir aynı)
review_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("app_id", IntegerType(), True),
    StructField("app_name", StringType(), True),
    StructField("review_text", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_votes", IntegerType(), True),
])

# Kafka'dan batch okuma
batch_raw_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:29092")
    .option("subscribe", "steam-reviews")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)

print(f"Kafka'dan okunan toplam kayıt: {batch_raw_df.count()}")
batch_raw_df.printSchema()

In [ ]:
# Batch - JSON Parse
batch_parsed_df = (
    batch_raw_df
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_timestamp")
    .select(
        from_json(col("json_str"), review_schema).alias("data"),
        col("kafka_timestamp")
    )
    .select("data.*", "kafka_timestamp")
)

print("Parsed Batch Schema:")
batch_parsed_df.printSchema()
batch_parsed_df.show(5, truncate=50)

In [ ]:
# Batch - Veri Temizleme
batch_cleaned_df = (
    batch_parsed_df
    .filter(col("review_text").isNotNull())
    .filter(length(trim(col("review_text"))) >= 10)
    .filter(col("review_score").isin(1, -1))
    .dropDuplicates(["app_id", "review_text"])
    .withColumn("label", when(col("review_score") == 1, 1).otherwise(0))
)

print(f"Temizleme sonrası kayıt sayısı: {batch_cleaned_df.count()}")
print("\nCleaned Batch Schema:")
batch_cleaned_df.printSchema()
batch_cleaned_df.show(5, truncate=50)

In [ ]:
# Batch - BRONZE Katmanı: Ham veri yazma
batch_parsed_df.write \
    .format("delta") \
    .mode("append") \
    .save("/delta/bronze/steam_reviews")

print("BRONZE katmanına yazıldı.")

# Doğrulama
bronze_read = spark.read.format("delta").load("/delta/bronze/steam_reviews")
print(f"Bronze kayıt sayısı: {bronze_read.count()}")
bronze_read.printSchema()
bronze_read.show(5, truncate=50)

In [ ]:
# Batch - SILVER Katmanı: Temizlenmiş veri yazma
batch_silver_df = batch_cleaned_df.select(
    "timestamp", "user_id", "event_type",
    "app_id", "app_name", "review_text",
    "review_score", "review_votes", "label"
)

batch_silver_df.write \
    .format("delta") \
    .mode("append") \
    .save("/delta/silver/steam_reviews")

print("SILVER katmanına yazıldı.")

# Doğrulama
silver_read = spark.read.format("delta").load("/delta/silver/steam_reviews")
print(f"Silver kayıt sayısı: {silver_read.count()}")
silver_read.printSchema()
silver_read.show(5, truncate=50)

In [ ]:
# Batch - GOLD Katmanı: Günlük oyun başına istatistikler
batch_gold_df = (
    batch_cleaned_df
    .withColumn("review_date", to_date(col("timestamp")))
    .groupBy("review_date", "app_id", "app_name")
    .agg(
        count("*").alias("total_reviews"),
        spark_sum(when(col("review_score") == 1, 1).otherwise(0)).alias("positive_count"),
        spark_sum(when(col("review_score") == -1, 1).otherwise(0)).alias("negative_count")
    )
)

batch_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/delta/gold/daily_stats")

print("GOLD katmanına yazıldı.")

# Doğrulama
gold_read = spark.read.format("delta").load("/delta/gold/daily_stats")
print(f"Gold kayıt sayısı: {gold_read.count()}")
gold_read.printSchema()
gold_read.orderBy(col("total_reviews").desc()).show(5, truncate=50)

In [ ]:
# Spark Session'ı kapat (notebook bitiminde)
# spark.stop()